# Advanced Policy Methods - A2C/A3C, SAC, TD3

This notebook bridges the gap between basic policy gradients (REINFORCE, Actor-Critic) and state-of-the-art continuous control algorithms used in robotics and complex environments.

## What We'll Learn

**Building on previous knowledge:**
- You've learned Q-learning, DQN, and basic policy gradients (REINFORCE, Actor-Critic, PPO)
- Those methods work great for discrete actions (left/right, up/down)
- But real-world control often needs **continuous actions** (steering angle, motor torque, etc.)

**This notebook covers three major advances:**

1. **A2C/A3C** - Asynchronous Advantage Actor-Critic
   - Parallel environment collection for faster training
   - Stabilizing policy gradients with synchronized updates

2. **SAC** - Soft Actor-Critic
   - Off-policy algorithm for continuous control
   - Maximum entropy RL: exploration built into the objective
   - Sample efficient and stable

3. **TD3** - Twin Delayed DDPG
   - Addresses overestimation bias in actor-critic methods
   - Uses twin critics and delayed policy updates
   - Simple but powerful improvements over DDPG

### Why These Methods Matter

- **A2C/A3C**: Foundation for distributed RL, used in AlphaGo and robotics
- **SAC**: State-of-the-art for continuous control, used in real-world robotics
- **TD3**: Simple improvements that made actor-critic methods competitive with SAC

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Normal, Categorical
import gymnasium as gym
import matplotlib.pyplot as plt
from collections import deque, namedtuple
import random
from typing import List, Tuple
import multiprocessing as mp
from copy import deepcopy

from aiml_notebooks import set_seed, get_device

# Set random seed for reproducibility
set_seed(42)

# Use CPU for RL (better compatibility)
device = get_device(prefer_cpu=True)
print(f"Using device: {device}")

## Part 1: A2C - Advantage Actor-Critic

### From Actor-Critic to A2C

In basic Actor-Critic, we update after every step. **A2C** (Advantage Actor-Critic) improves this by:

1. **Collecting rollouts**: Run multiple steps before updating
2. **Computing advantages**: Use n-step returns for lower variance
3. **Synchronous updates**: Update after collecting fixed-length trajectories

### The A2C Algorithm

```
For each iteration:
    1. Collect n steps of experience (states, actions, rewards)
    2. Compute n-step returns: R_t = r_t + γr_{t+1} + ... + γ^n V(s_{t+n})
    3. Compute advantages: A_t = R_t - V(s_t)
    4. Update actor: maximize log π(a_t|s_t) * A_t
    5. Update critic: minimize (V(s_t) - R_t)²
```

**Key insight**: Collecting multiple steps creates a more stable gradient estimate than single-step updates.

### Environment: CartPole

We'll start with **CartPole-v1** to understand the algorithms, then move to continuous control.

In [ ]:
# Create environment
env = gym.make('CartPole-v1')

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

print(f"State dimension: {state_dim}")
print(f"Action dimension: {action_dim}")
print(f"Action space: Discrete (left/right)")
print(f"Goal: Achieve average reward of 475+ over 100 episodes")

### A2C Networks

We'll use **shared backbone** architecture:
- Common feature extractor for both actor and critic
- Separate heads for policy and value
- More parameter efficient and faster training

In [ ]:
class A2CNetwork(nn.Module):
    """A2C network with shared backbone for actor and critic."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        
        # Shared backbone
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Actor head (policy)
        self.actor = nn.Linear(hidden_dim, action_dim)
        
        # Critic head (value function)
        self.critic = nn.Linear(hidden_dim, 1)
    
    def forward(self, state):
        """Forward pass returns both policy logits and value."""
        features = self.backbone(state)
        logits = self.actor(features)
        value = self.critic(features)
        return logits, value
    
    def get_action(self, state):
        """Sample action from policy."""
        logits, value = self.forward(state)
        dist = Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob, value

# Test network
a2c_net = A2CNetwork(state_dim, action_dim).to(device)
test_state = torch.randn(1, state_dim).to(device)
logits, value = a2c_net(test_state)

print(f"Network output shapes:")
print(f"  Logits (policy): {logits.shape}")
print(f"  Value: {value.shape}")
print(f"\nTotal parameters: {sum(p.numel() for p in a2c_net.parameters())}")

### Computing N-Step Returns

N-step returns bridge Monte Carlo (full episode) and TD (one-step):

$$R_t^{(n)} = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + ... + \gamma^{n-1} r_{t+n-1} + \gamma^n V(s_{t+n})$$

**Benefits:**
- Lower variance than Monte Carlo (bootstraps from value function)
- Lower bias than 1-step TD (uses more actual rewards)
- Tunable via n (typically n=5 or n=10)

In [ ]:
def compute_n_step_returns(rewards, values, next_value, gamma=0.99, n_steps=5):
    """Compute n-step returns for advantage estimation.
    
    Args:
        rewards: List of rewards [r_0, r_1, ..., r_{T-1}]
        values: List of value estimates [V(s_0), V(s_1), ..., V(s_{T-1})]
        next_value: V(s_T) - value of final state
        gamma: Discount factor
        n_steps: Number of steps for n-step return
    
    Returns:
        returns: List of n-step returns
    """
    T = len(rewards)
    returns = []
    
    for t in range(T):
        # Compute n-step return
        n_step_return = 0
        
        # Sum discounted rewards up to n steps or end of episode
        for k in range(n_steps):
            if t + k < T:
                n_step_return += (gamma ** k) * rewards[t + k]
            else:
                break
        
        # Bootstrap from value function
        if t + n_steps < T:
            n_step_return += (gamma ** n_steps) * values[t + n_steps]
        else:
            # If we reach the end, use next_value
            n_step_return += (gamma ** (T - t)) * next_value
        
        returns.append(n_step_return)
    
    return returns

# Test n-step returns
test_rewards = [1.0, 1.0, 1.0, 1.0, 1.0]
test_values = [5.0, 4.0, 3.0, 2.0, 1.0]
test_next_value = 0.0

returns_1step = compute_n_step_returns(test_rewards, test_values, test_next_value, n_steps=1)
returns_5step = compute_n_step_returns(test_rewards, test_values, test_next_value, n_steps=5)

print("Example with rewards=[1,1,1,1,1], values=[5,4,3,2,1]:")
print(f"  1-step returns: {[f'{r:.2f}' for r in returns_1step]}")
print(f"  5-step returns: {[f'{r:.2f}' for r in returns_5step]}")
print("\n→ N-step returns use more actual rewards before bootstrapping")

### A2C Agent Implementation

In [ ]:
class A2CAgent:
    """A2C agent with n-step returns."""
    
    def __init__(self, network, lr=1e-3, gamma=0.99, n_steps=5, 
                 value_coef=0.5, entropy_coef=0.01):
        self.network = network
        self.optimizer = optim.Adam(network.parameters(), lr=lr)
        self.gamma = gamma
        self.n_steps = n_steps
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        
        # Storage for rollout
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.values = []
    
    def select_action(self, state):
        """Select action and store experience."""
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        action, log_prob, value = self.network.get_action(state_tensor)
        
        self.states.append(state)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.values.append(value.item())
        
        return action
    
    def store_reward(self, reward):
        """Store reward."""
        self.rewards.append(reward)
    
    def update(self, next_state, done):
        """Update networks using collected rollout."""
        # Get value of next state for bootstrapping
        if done:
            next_value = 0.0
        else:
            with torch.no_grad():
                next_state_tensor = torch.FloatTensor(next_state).unsqueeze(0).to(device)
                _, next_value = self.network(next_state_tensor)
                next_value = next_value.item()
        
        # Compute n-step returns
        returns = compute_n_step_returns(
            self.rewards, self.values, next_value, 
            self.gamma, self.n_steps
        )
        
        # Convert to tensors
        states = torch.FloatTensor(np.array(self.states)).to(device)
        actions = torch.LongTensor(self.actions).to(device)
        returns = torch.FloatTensor(returns).to(device)
        old_log_probs = torch.stack(self.log_probs)
        old_values = torch.FloatTensor(self.values).to(device)
        
        # Forward pass to get current policy and values
        logits, values = self.network(states)
        values = values.squeeze()
        
        # Compute advantages
        advantages = returns - old_values
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        # Policy loss (actor)
        dist = Categorical(logits=logits)
        log_probs = dist.log_prob(actions)
        policy_loss = -(log_probs * advantages.detach()).mean()
        
        # Value loss (critic)
        value_loss = F.mse_loss(values, returns)
        
        # Entropy bonus for exploration
        entropy = dist.entropy().mean()
        
        # Total loss
        loss = policy_loss + self.value_coef * value_loss - self.entropy_coef * entropy
        
        # Optimize
        self.optimizer.zero_grad()
        loss.backward()
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(self.network.parameters(), 0.5)
        self.optimizer.step()
        
        # Clear rollout
        self.states = []
        self.actions = []
        self.log_probs = []
        self.rewards = []
        self.values = []
        
        return loss.item(), policy_loss.item(), value_loss.item(), entropy.item()

# Create A2C agent
a2c_network = A2CNetwork(state_dim, action_dim).to(device)
a2c_agent = A2CAgent(a2c_network, lr=1e-3, n_steps=5)
print("A2C agent created!")

### Training A2C

A2C updates after collecting fixed-length rollouts (e.g., every 5 steps).

In [ ]:
def train_a2c(agent, env, num_episodes=300, update_freq=5, print_every=50):
    """Train A2C agent."""
    episode_rewards = []
    running_reward = deque(maxlen=100)
    
    for episode in range(num_episodes):
        state, _ = env.reset()
        episode_reward = 0
        done = False
        steps = 0
        
        while not done:
            # Select action
            action = agent.select_action(state)
            
            # Take step
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            agent.store_reward(reward)
            episode_reward += reward
            steps += 1
            
            # Update every update_freq steps or at episode end
            if steps % update_freq == 0 or done:
                agent.update(next_state, done)
            
            state = next_state
        
        episode_rewards.append(episode_reward)
        running_reward.append(episode_reward)
        
        if (episode + 1) % print_every == 0:
            avg_reward = np.mean(running_reward)
            print(f"Episode {episode+1}/{num_episodes} | Avg Reward: {avg_reward:.2f}")
            
            if avg_reward > 475:
                print(f"\n✓ Solved in {episode+1} episodes!")
                break
    
    return episode_rewards

print("Training A2C agent...\n")
a2c_rewards = train_a2c(a2c_agent, env, num_episodes=300, update_freq=5)

### A2C Learning Curve

In [ ]:
def plot_learning_curve(rewards, title="Training Progress", window=10):
    """Plot episode rewards with moving average."""
    plt.figure(figsize=(10, 5))
    plt.plot(rewards, alpha=0.3, color='blue', label='Episode Reward')
    
    if len(rewards) >= window:
        moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')
        plt.plot(range(window-1, len(rewards)), moving_avg, 
                color='blue', linewidth=2, label=f'{window}-Episode Avg')
    
    plt.axhline(y=475, color='green', linestyle='--', alpha=0.7, label='Success Threshold')
    plt.xlabel('Episode', fontsize=11)
    plt.ylabel('Total Reward', fontsize=11)
    plt.title(title, fontsize=13, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_learning_curve(a2c_rewards, title="A2C Training on CartPole-v1")

### Understanding A3C (Asynchronous A2C)

**A3C** extends A2C with **parallel environment collection**:

```
Global Network (shared parameters)
    ↓         ↓         ↓         ↓
Worker 1  Worker 2  Worker 3  Worker 4  (parallel environments)
    ↓         ↓         ↓         ↓
Gradients → Asynchronous updates → Global Network
```

**Benefits:**
1. **Faster training**: Multiple environments collect data in parallel
2. **Decorrelated samples**: Different workers explore different states
3. **No replay buffer needed**: Parallel collection provides diversity

**Why we're not implementing it here:**
- A3C requires multiprocessing which adds complexity
- Modern practice uses **synchronous** parallel collection (like in PPO)
- A2C is easier to understand and implement

**Key insight**: A3C showed that parallel environment collection + simple actor-critic = powerful RL algorithm. This inspired modern distributed RL systems.

## Part 2: Continuous Control - Pendulum Environment

### The Challenge of Continuous Actions

So far we've used **discrete actions** (left/right). Real-world control needs **continuous actions**:
- Robot joint angles: -π to +π radians
- Motor torque: -2.0 to +2.0 Newton-meters
- Steering angle: -45° to +45°

**Solution**: Instead of outputting action probabilities, output parameters of a **continuous distribution**:
- Mean μ and standard deviation σ
- Sample action from Normal(μ, σ)

Let's use **Pendulum-v1** - swing up an inverted pendulum.

In [ ]:
# Create continuous control environment
env_continuous = gym.make('Pendulum-v1')

state_dim_cont = env_continuous.observation_space.shape[0]
action_dim_cont = env_continuous.action_space.shape[0]
action_low = env_continuous.action_space.low[0]
action_high = env_continuous.action_space.high[0]

print(f"State dimension: {state_dim_cont}")
print(f"Action dimension: {action_dim_cont}")
print(f"Action range: [{action_low:.1f}, {action_high:.1f}]")
print(f"\nState: [cos(θ), sin(θ), angular_velocity]")
print(f"Action: torque applied to pendulum")
print(f"Goal: Keep pendulum upright (θ=0)")

### Test Random Policy

Let's see how a random policy performs.

In [ ]:
def test_random_policy_continuous(env, num_episodes=10):
    """Test random policy on continuous action environment."""
    rewards = []
    
    for _ in range(num_episodes):
        state, _ = env.reset()
        episode_reward = 0
        done = False
        
        while not done:
            action = env.action_space.sample()
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_reward += reward
        
        rewards.append(episode_reward)
    
    return rewards

random_rewards = test_random_policy_continuous(env_continuous, num_episodes=100)
print(f"Random policy: {np.mean(random_rewards):.2f} ± {np.std(random_rewards):.2f}")
print(f"(Rewards are negative - closer to 0 is better)")
print(f"Good policy: -200 to -150")

## Part 3: SAC - Soft Actor-Critic

### Maximum Entropy Reinforcement Learning

**Standard RL objective**: Maximize expected return

$$J(\pi) = \mathbb{E}\left[\sum_{t=0}^T r_t\right]$$

**SAC objective**: Maximize return AND entropy

$$J(\pi) = \mathbb{E}\left[\sum_{t=0}^T \left(r_t + \alpha \mathcal{H}(\pi(\cdot|s_t))\right)\right]$$

where $\mathcal{H}(\pi) = -\mathbb{E}_{a \sim \pi}[\log \pi(a|s)]$ is the entropy.

**Why entropy?**
1. **Exploration**: High entropy = policy is uncertain = explores more
2. **Robustness**: Prevents premature convergence to suboptimal policies
3. **Multi-modal**: Can learn multiple good strategies

### SAC Architecture

SAC has **three networks**:
1. **Actor** (policy): π(a|s) - stochastic Gaussian policy
2. **Critic** (Q-function): Q(s,a) - action-value function
3. **Target critic**: Q'(s,a) - slowly updated copy (like DQN)

**Key features:**
- **Off-policy**: Uses replay buffer (like DQN)
- **Twin critics**: Uses two Q-networks to reduce overestimation
- **Automatic temperature tuning**: Learns α automatically

### SAC Actor Network

The actor outputs mean μ and log(σ) for a Gaussian distribution:

$$a \sim \mathcal{N}(\mu(s), \sigma(s))$$

We use **tanh** squashing to bound actions to [-1, 1].

In [ ]:
class SACActor(nn.Module):
    """SAC actor network for continuous actions."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=256, log_std_min=-20, log_std_max=2):
        super().__init__()
        self.log_std_min = log_std_min
        self.log_std_max = log_std_max
        
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        self.mean = nn.Linear(hidden_dim, action_dim)
        self.log_std = nn.Linear(hidden_dim, action_dim)
    
    def forward(self, state):
        """Forward pass returns action distribution parameters."""
        x = self.net(state)
        mean = self.mean(x)
        log_std = self.log_std(x)
        log_std = torch.clamp(log_std, self.log_std_min, self.log_std_max)
        return mean, log_std
    
    def sample(self, state):
        """Sample action from policy."""
        mean, log_std = self.forward(state)
        std = log_std.exp()
        
        # Sample from Normal distribution
        normal = Normal(mean, std)
        x_t = normal.rsample()  # Reparameterization trick
        
        # Squash to [-1, 1] using tanh
        action = torch.tanh(x_t)
        
        # Compute log probability (with correction for tanh squashing)
        log_prob = normal.log_prob(x_t)
        log_prob -= torch.log(1 - action.pow(2) + 1e-6)
        log_prob = log_prob.sum(1, keepdim=True)
        
        return action, log_prob, torch.tanh(mean)

# Test actor
sac_actor = SACActor(state_dim_cont, action_dim_cont).to(device)
test_state = torch.randn(1, state_dim_cont).to(device)
action, log_prob, mean_action = sac_actor.sample(test_state)

print(f"Actor output shapes:")
print(f"  Sampled action: {action.shape}")
print(f"  Log probability: {log_prob.shape}")
print(f"  Mean action: {mean_action.shape}")
print(f"\nSample action value: {action.item():.3f} (bounded to [-1, 1])")

### SAC Critic Network

The critic estimates Q(s,a). SAC uses **two critics** (twin Q-networks) to reduce overestimation bias.

In [ ]:
class SACCritic(nn.Module):
    """SAC critic network (Q-function)."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        
        # Q1 network
        self.q1 = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        
        # Q2 network
        self.q2 = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, state, action):
        """Forward pass returns both Q-values."""
        x = torch.cat([state, action], dim=1)
        q1 = self.q1(x)
        q2 = self.q2(x)
        return q1, q2

# Test critic
sac_critic = SACCritic(state_dim_cont, action_dim_cont).to(device)
test_action = torch.randn(1, action_dim_cont).to(device)
q1, q2 = sac_critic(test_state, test_action)

print(f"Critic output shapes:")
print(f"  Q1: {q1.shape}")
print(f"  Q2: {q2.shape}")
print(f"\n→ Twin critics help reduce overestimation bias")

### Replay Buffer for SAC

SAC is **off-policy** and uses a replay buffer like DQN.

In [ ]:
class ReplayBuffer:
    """Replay buffer for off-policy algorithms."""
    
    def __init__(self, capacity=100000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        
        return (
            torch.FloatTensor(np.array(states)),
            torch.FloatTensor(np.array(actions)),
            torch.FloatTensor(rewards).unsqueeze(1),
            torch.FloatTensor(np.array(next_states)),
            torch.FloatTensor(dones).unsqueeze(1)
        )
    
    def __len__(self):
        return len(self.buffer)

# Test buffer
buffer = ReplayBuffer(capacity=1000)
for _ in range(100):
    buffer.push(
        np.random.randn(state_dim_cont),
        np.random.randn(action_dim_cont),
        np.random.randn(),
        np.random.randn(state_dim_cont),
        False
    )

states, actions, rewards, next_states, dones = buffer.sample(32)
print(f"Batch shapes:")
print(f"  States: {states.shape}")
print(f"  Actions: {actions.shape}")
print(f"  Rewards: {rewards.shape}")
print(f"  Next states: {next_states.shape}")
print(f"  Dones: {dones.shape}")

### SAC Agent

The SAC agent coordinates actor, critics, and target critics.

In [ ]:
class SACAgent:
    """Soft Actor-Critic agent."""
    
    def __init__(self, state_dim, action_dim, device,
                 actor_lr=3e-4, critic_lr=3e-4, gamma=0.99, tau=0.005, alpha=0.2):
        self.device = device
        self.gamma = gamma
        self.tau = tau
        self.alpha = alpha
        
        # Actor
        self.actor = SACActor(state_dim, action_dim).to(device)
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=actor_lr)
        
        # Critics
        self.critic = SACCritic(state_dim, action_dim).to(device)
        self.critic_target = SACCritic(state_dim, action_dim).to(device)
        self.critic_target.load_state_dict(self.critic.state_dict())
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=critic_lr)
        
        # Replay buffer
        self.replay_buffer = ReplayBuffer()
    
    def select_action(self, state, eval=False):
        """Select action from policy."""
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            if eval:
                _, _, action = self.actor.sample(state)
            else:
                action, _, _ = self.actor.sample(state)
        return action.cpu().numpy()[0]
    
    def update(self, batch_size=256):
        """Update actor and critics."""
        if len(self.replay_buffer) < batch_size:
            return None, None
        
        # Sample batch
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(batch_size)
        states = states.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_states = next_states.to(self.device)
        dones = dones.to(self.device)
        
        # Update critics
        with torch.no_grad():
            # Sample actions from current policy for next states
            next_actions, next_log_probs, _ = self.actor.sample(next_states)
            
            # Compute target Q-values (use minimum of two critics)
            q1_next, q2_next = self.critic_target(next_states, next_actions)
            q_next = torch.min(q1_next, q2_next)
            
            # Add entropy bonus
            q_target = rewards + (1 - dones) * self.gamma * (q_next - self.alpha * next_log_probs)
        
        # Current Q-values
        q1, q2 = self.critic(states, actions)
        
        # Critic loss
        critic_loss = F.mse_loss(q1, q_target) + F.mse_loss(q2, q_target)
        
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        
        # Update actor
        actions_new, log_probs, _ = self.actor.sample(states)
        q1_new, q2_new = self.critic(states, actions_new)
        q_new = torch.min(q1_new, q2_new)
        
        # Actor loss (maximize Q - α*entropy)
        actor_loss = (self.alpha * log_probs - q_new).mean()
        
        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()
        
        # Soft update target network
        for target_param, param in zip(self.critic_target.parameters(), self.critic.parameters()):
            target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
        
        return actor_loss.item(), critic_loss.item()

# Create SAC agent
sac_agent = SACAgent(state_dim_cont, action_dim_cont, device)
print("SAC agent created!")
print(f"Total parameters: {sum(p.numel() for p in sac_agent.actor.parameters()) + sum(p.numel() for p in sac_agent.critic.parameters())}")

### Training SAC

SAC training involves:
1. Collecting experience with current policy
2. Storing in replay buffer
3. Sampling batches and updating networks

In [ ]:
def train_sac(agent, env, num_episodes=150, batch_size=256, 
              start_steps=1000, update_every=1, print_every=25):
    """Train SAC agent."""
    episode_rewards = []
    total_steps = 0
    
    for episode in range(num_episodes):
        state, _ = env.reset()
        episode_reward = 0
        done = False
        
        while not done:
            # Select action (random for initial exploration)
            if total_steps < start_steps:
                action = env.action_space.sample()
            else:
                action = agent.select_action(state)
            
            # Take step
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Store transition
            agent.replay_buffer.push(state, action, reward, next_state, done)
            
            # Update networks
            if total_steps >= start_steps and total_steps % update_every == 0:
                agent.update(batch_size)
            
            state = next_state
            episode_reward += reward
            total_steps += 1
        
        episode_rewards.append(episode_reward)
        
        if (episode + 1) % print_every == 0:
            avg_reward = np.mean(episode_rewards[-print_every:])
            print(f"Episode {episode+1}/{num_episodes} | "
                  f"Avg Reward: {avg_reward:.2f} | "
                  f"Steps: {total_steps}")
    
    return episode_rewards

print("Training SAC agent on Pendulum-v1...\n")
print("(This may take a few minutes)\n")
sac_rewards = train_sac(sac_agent, env_continuous, num_episodes=150, start_steps=1000)

### SAC Learning Curve

In [ ]:
plot_learning_curve(sac_rewards, title="SAC Training on Pendulum-v1", window=10)

## Part 4: TD3 - Twin Delayed DDPG

### Improving Actor-Critic with TD3

**DDPG** (Deep Deterministic Policy Gradient) is the deterministic version of actor-critic for continuous actions. However, it suffers from:
1. **Overestimation bias**: Q-values get too optimistic
2. **High variance**: Updates can be unstable
3. **Poor exploration**: Deterministic policy doesn't explore enough

**TD3** fixes these with three simple tricks:

### 1. Twin Critics (Clipped Double Q-Learning)

Use two Q-networks and take the **minimum** for target computation:

$$y = r + \gamma \min_{i=1,2} Q_{\theta_i}(s', \pi(s'))$$

**Why?** Taking the minimum reduces overestimation bias.

### 2. Delayed Policy Updates

Update actor **less frequently** than critics (e.g., every 2 critic updates).

**Why?** Gives critics time to converge before updating policy.

### 3. Target Policy Smoothing

Add noise to target actions:

$$y = r + \gamma Q(s', \pi(s') + \epsilon), \quad \epsilon \sim \mathcal{N}(0, \sigma)$$

**Why?** Smooths Q-values and reduces variance in target computation.

### TD3 Networks

TD3 uses **deterministic** policy (unlike SAC's stochastic policy).

In [ ]:
class TD3Actor(nn.Module):
    """TD3 deterministic actor."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=256, max_action=1.0):
        super().__init__()
        self.max_action = max_action
        
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim),
            nn.Tanh()
        )
    
    def forward(self, state):
        """Deterministic action (no sampling)."""
        return self.max_action * self.net(state)

class TD3Critic(nn.Module):
    """TD3 twin critic."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        
        # Q1
        self.q1 = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        
        # Q2
        self.q2 = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, state, action):
        x = torch.cat([state, action], dim=1)
        return self.q1(x), self.q2(x)
    
    def q1_forward(self, state, action):
        """Only Q1 (for actor update)."""
        x = torch.cat([state, action], dim=1)
        return self.q1(x)

# Test networks
td3_actor = TD3Actor(state_dim_cont, action_dim_cont, max_action=2.0).to(device)
td3_critic = TD3Critic(state_dim_cont, action_dim_cont).to(device)

test_state = torch.randn(1, state_dim_cont).to(device)
action = td3_actor(test_state)
q1, q2 = td3_critic(test_state, action)

print(f"TD3 Actor output: {action.shape} (deterministic)")
print(f"TD3 Critic outputs: Q1={q1.shape}, Q2={q2.shape}")

### TD3 Agent

In [ ]:
class TD3Agent:
    """Twin Delayed DDPG agent."""
    
    def __init__(self, state_dim, action_dim, device, max_action=2.0,
                 actor_lr=3e-4, critic_lr=3e-4, gamma=0.99, tau=0.005,
                 policy_noise=0.2, noise_clip=0.5, policy_delay=2):
        self.device = device
        self.max_action = max_action
        self.gamma = gamma
        self.tau = tau
        self.policy_noise = policy_noise
        self.noise_clip = noise_clip
        self.policy_delay = policy_delay
        self.total_it = 0
        
        # Actor
        self.actor = TD3Actor(state_dim, action_dim, max_action=max_action).to(device)
        self.actor_target = TD3Actor(state_dim, action_dim, max_action=max_action).to(device)
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=actor_lr)
        
        # Critic
        self.critic = TD3Critic(state_dim, action_dim).to(device)
        self.critic_target = TD3Critic(state_dim, action_dim).to(device)
        self.critic_target.load_state_dict(self.critic.state_dict())
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=critic_lr)
        
        # Replay buffer
        self.replay_buffer = ReplayBuffer()
    
    def select_action(self, state, noise=0.1):
        """Select action with optional exploration noise."""
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        action = self.actor(state).cpu().data.numpy()[0]
        
        if noise > 0:
            action += np.random.normal(0, noise, size=action.shape)
            action = np.clip(action, -self.max_action, self.max_action)
        
        return action
    
    def update(self, batch_size=256):
        """Update actor and critics."""
        if len(self.replay_buffer) < batch_size:
            return None, None
        
        self.total_it += 1
        
        # Sample batch
        states, actions, rewards, next_states, dones = self.replay_buffer.sample(batch_size)
        states = states.to(self.device)
        actions = actions.to(self.device)
        rewards = rewards.to(self.device)
        next_states = next_states.to(self.device)
        dones = dones.to(self.device)
        
        with torch.no_grad():
            # Target policy smoothing: add noise to target actions
            noise = (torch.randn_like(actions) * self.policy_noise).clamp(
                -self.noise_clip, self.noise_clip
            )
            next_actions = (self.actor_target(next_states) + noise).clamp(
                -self.max_action, self.max_action
            )
            
            # Clipped double Q-learning: use minimum of two critics
            q1_next, q2_next = self.critic_target(next_states, next_actions)
            q_next = torch.min(q1_next, q2_next)
            q_target = rewards + (1 - dones) * self.gamma * q_next
        
        # Update critics
        q1, q2 = self.critic(states, actions)
        critic_loss = F.mse_loss(q1, q_target) + F.mse_loss(q2, q_target)
        
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        
        actor_loss = None
        
        # Delayed policy update
        if self.total_it % self.policy_delay == 0:
            # Update actor
            actor_loss = -self.critic.q1_forward(states, self.actor(states)).mean()
            
            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()
            
            # Soft update target networks
            for target_param, param in zip(self.actor_target.parameters(), self.actor.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
            
            for target_param, param in zip(self.critic_target.parameters(), self.critic.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
            
            actor_loss = actor_loss.item()
        
        return actor_loss, critic_loss.item()

# Create TD3 agent
td3_agent = TD3Agent(state_dim_cont, action_dim_cont, device, max_action=2.0)
print("TD3 agent created!")

### Training TD3

In [ ]:
def train_td3(agent, env, num_episodes=150, batch_size=256, 
              start_steps=1000, expl_noise=0.1, print_every=25):
    """Train TD3 agent."""
    episode_rewards = []
    total_steps = 0
    
    for episode in range(num_episodes):
        state, _ = env.reset()
        episode_reward = 0
        done = False
        
        while not done:
            # Select action
            if total_steps < start_steps:
                action = env.action_space.sample()
            else:
                action = agent.select_action(state, noise=expl_noise)
            
            # Take step
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            
            # Store transition
            agent.replay_buffer.push(state, action, reward, next_state, done)
            
            # Update networks
            if total_steps >= start_steps:
                agent.update(batch_size)
            
            state = next_state
            episode_reward += reward
            total_steps += 1
        
        episode_rewards.append(episode_reward)
        
        if (episode + 1) % print_every == 0:
            avg_reward = np.mean(episode_rewards[-print_every:])
            print(f"Episode {episode+1}/{num_episodes} | "
                  f"Avg Reward: {avg_reward:.2f} | "
                  f"Steps: {total_steps}")
    
    return episode_rewards

print("Training TD3 agent on Pendulum-v1...\n")
print("(This may take a few minutes)\n")
td3_rewards = train_td3(td3_agent, env_continuous, num_episodes=150, start_steps=1000)

### TD3 Learning Curve

In [ ]:
plot_learning_curve(td3_rewards, title="TD3 Training on Pendulum-v1", window=10)

## Comparison: A2C vs SAC vs TD3

In [ ]:
# Compare SAC and TD3 on Pendulum
fig, ax = plt.subplots(figsize=(12, 6))

window = 10

# SAC
if len(sac_rewards) >= window:
    sac_smooth = np.convolve(sac_rewards, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(sac_rewards)), sac_smooth, 
            label='SAC', linewidth=2, color='blue')

# TD3
if len(td3_rewards) >= window:
    td3_smooth = np.convolve(td3_rewards, np.ones(window)/window, mode='valid')
    ax.plot(range(window-1, len(td3_rewards)), td3_smooth, 
            label='TD3', linewidth=2, color='red')

# Random baseline
ax.axhline(y=np.mean(random_rewards), color='gray', linestyle='--', 
          alpha=0.7, label='Random Policy')

ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Average Reward', fontsize=12)
ax.set_title('SAC vs TD3 on Pendulum-v1', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nFinal Performance (last 20 episodes):")
print(f"  SAC: {np.mean(sac_rewards[-20:]):.2f}")
print(f"  TD3: {np.mean(td3_rewards[-20:]):.2f}")
print(f"  Random: {np.mean(random_rewards):.2f}")

## Key Takeaways

### Algorithm Comparison

| Algorithm | Policy Type | Action Space | Sample Efficiency | Key Innovation |
|-----------|-------------|--------------|-------------------|----------------|
| **A2C** | Stochastic | Discrete/Continuous | Medium | N-step returns, synchronous updates |
| **A3C** | Stochastic | Discrete/Continuous | Medium | Parallel workers, asynchronous |
| **SAC** | Stochastic | Continuous | High | Maximum entropy, off-policy |
| **TD3** | Deterministic | Continuous | High | Twin critics, delayed updates |

### When to Use Each

**A2C/A3C:**
- ✓ Good for discrete actions
- ✓ Works with on-policy data
- ✓ Simple and stable
- ✗ Not as sample efficient as off-policy methods

**SAC:**
- ✓ State-of-the-art for continuous control
- ✓ Very sample efficient (off-policy + replay buffer)
- ✓ Built-in exploration (maximum entropy)
- ✓ Robust hyperparameters
- ✗ More complex than TD3

**TD3:**
- ✓ Simple improvements over DDPG
- ✓ Very competitive with SAC
- ✓ Deterministic policy (easier to deploy)
- ✗ Requires manual exploration noise tuning

### Core Concepts

✓ **N-step returns** reduce variance while maintaining low bias

✓ **Off-policy + replay buffer** = sample efficient learning

✓ **Twin critics** reduce overestimation bias

✓ **Maximum entropy** encourages exploration automatically

✓ **Delayed updates** give critics time to converge

✓ **Target networks** stabilize training

## Advanced Topics & Extensions

### 1. Distributional RL
Instead of learning E[Q(s,a)], learn the full distribution of returns. Algorithms: C51, QR-DQN, IQN.

### 2. Model-Based RL
Learn a model of the environment to simulate future trajectories. Algorithms: Dreamer, MBPO, World Models.

### 3. Multi-Agent RL
Extend to multiple agents cooperating or competing. Challenges: non-stationarity, credit assignment.

### 4. Offline RL
Learn from fixed datasets without environment interaction. Algorithms: CQL, IQL, Decision Transformers.

### 5. Hierarchical RL
Learn policies at multiple time scales (high-level goals + low-level actions). Algorithms: HAC, HIRO.

### 6. Meta-RL
Learn to adapt quickly to new tasks. Algorithms: MAML, RL², Meta-World benchmarks.

### Real-World Applications

- **Robotics**: SAC/TD3 for manipulation, locomotion
- **Autonomous Driving**: Policy gradients for trajectory planning
- **Game Playing**: A3C for Atari, AlphaGo used policy gradients
- **Resource Management**: Datacenter cooling, traffic light control
- **Finance**: Portfolio optimization, trading strategies

## Summary

You've learned three major advances in policy-based RL:

1. **A2C/A3C** - Using n-step returns and parallel collection for stable on-policy learning
2. **SAC** - Maximum entropy RL with automatic exploration for sample-efficient continuous control
3. **TD3** - Simple but powerful improvements (twin critics, delayed updates) over DDPG

**The progression:**
- Basic policy gradients (REINFORCE) → A2C (n-step returns) → A3C (parallel workers)
- Actor-Critic → DDPG (continuous, deterministic) → TD3 (twin critics)
- Off-policy + maximum entropy → SAC (state-of-the-art)

**Modern RL landscape (2024):**
- **Discrete actions**: PPO (most popular), A2C
- **Continuous control**: SAC, TD3 (both excellent)
- **Robotics**: SAC is standard, TD3 close second
- **Research**: Exploring offline RL, model-based RL, multi-task learning

You now have the foundation to tackle real-world continuous control problems!